# Play Store Reviews & Installs Analysis (Task-3)
This notebook implements the analysis and plotting of total cumulative installs over time segmented by category, with filters applied, category translations, and growth shading.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Set style and font to support Devanagari and Tamil
plt.style.use('dark_background')
plt.rcParams['font.family'] = 'Nirmala UI'
plt.rcParams['axes.unicode_minus'] = False

## 1. Load and Clean Dataset
We load from `Dataset/` subdirectory and perform standard cleaning on Installs and Reviews.

In [ ]:
apps_df = pd.read_csv('Dataset/Play Store Data.csv')

# Clean Installs
apps_df['Installs'] = apps_df['Installs'].astype(str).str.replace(',', '').str.replace('+', '').str.strip()
apps_df = apps_df[apps_df['Installs'].str.isnumeric()]
apps_df['Installs'] = apps_df['Installs'].astype(float)

# Clean Reviews
apps_df['Reviews'] = pd.to_numeric(apps_df['Reviews'], errors='coerce')
apps_df = apps_df.dropna(subset=['Reviews'])

# Clean Dates
apps_df['Last Updated'] = pd.to_datetime(apps_df['Last Updated'], errors='coerce')
apps_df = apps_df.dropna(subset=['Last Updated'])

print("Initial dataset loaded. Shape:", apps_df.shape)

## 2. Apply Custom Filters
We filter based on the following constraints:
- App name must not start with 'x', 'y', 'z' (case-insensitive).
- App name must not contain the letter 'S' (case-insensitive).
- App category must start with 'E', 'C', 'B' OR be 'DATING'.
- Reviews must be more than 500.

In [ ]:
# Filter out App names starting with x, y, z
apps_df = apps_df[~apps_df['App'].str.lower().str.startswith(('x', 'y', 'z'), na=False)]

# Filter out App names containing 'S'
apps_df = apps_df[~apps_df['App'].str.contains('S', case=False, na=False)]

# Filter reviews > 500
apps_df = apps_df[apps_df['Reviews'] > 500]

# Filter categories starting with E, C, B or equal to DATING
categories_to_keep = [cat for cat in apps_df['Category'].unique() if cat.startswith(('E', 'C', 'B'))] + ['DATING']
apps_df = apps_df[apps_df['Category'].isin(categories_to_keep)]

print("Filtered dataset shape:", apps_df.shape)
print("Remaining categories:", apps_df['Category'].unique())

## 3. Re-sample to Monthly Frequency and Compute Growth
We resample to regular monthly periods, calculate cumulative installs, and compute MoM growth rates.

In [ ]:
apps_df['YearMonth'] = apps_df['Last Updated'].dt.to_period('M')

# Group by Category and Month, sum installs
grouped = apps_df.groupby(['Category', 'YearMonth'])['Installs'].sum().reset_index()
grouped = grouped.sort_values(['Category', 'YearMonth'])

min_period = grouped['YearMonth'].min()
max_period = grouped['YearMonth'].max()
all_months = pd.period_range(start=min_period, end=max_period, freq='M')

resampled_list = []
for cat in grouped['Category'].unique():
    cat_df = grouped[grouped['Category'] == cat].set_index('YearMonth')
    cat_df = cat_df.reindex(all_months, fill_value=0)
    cat_df['Category'] = cat
    cat_df = cat_df.reset_index().rename(columns={'index': 'YearMonth'})
    
    # Cumulative installs over time
    cat_df['Cumulative_Installs'] = cat_df['Installs'].cumsum()
    
    # MoM Growth rate
    cat_df['Prev_Cumulative'] = cat_df['Cumulative_Installs'].shift(1)
    cat_df['MoM_Growth'] = (cat_df['Cumulative_Installs'] - cat_df['Prev_Cumulative']) / cat_df['Prev_Cumulative']
    cat_df['MoM_Growth'] = cat_df['MoM_Growth'].replace([np.inf, -np.inf], np.nan).fillna(0)
    resampled_list.append(cat_df)

resampled_df = pd.concat(resampled_list)
resampled_df['Date'] = resampled_df['YearMonth'].dt.to_timestamp()
print("Resampled dataset compiled.")

## 4. Plot Time-Series and Save Graph
We display translations on the legend and highlight significant growth (>20% MoM) by shading under the curve.

In [ ]:
translation_map = {
    'BEAUTY': 'सौंदर्य',
    'BUSINESS': 'வணிகம்',
    'DATING': 'Verabredung'
}

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_yscale('log')

for cat in resampled_df['Category'].unique():
    cat_data = resampled_df[resampled_df['Category'] == cat].sort_values('Date')
    cat_data = cat_data[cat_data['Cumulative_Installs'] > 0]
    if len(cat_data) == 0:
        continue
    label_name = translation_map.get(cat, cat)
    
    # Plot category line
    line, = ax.plot(cat_data['Date'], cat_data['Cumulative_Installs'], label=label_name, linewidth=2)
    color = line.get_color()
    
    # Shade regions where MoM growth > 20%
    dates = cat_data['Date'].values
    installs = cat_data['Cumulative_Installs'].values
    growth = cat_data['MoM_Growth'].values
    
    for i in range(1, len(dates)):
        if growth[i] > 0.20:
            ax.fill_between(
                dates[i-1:i+1],
                installs[i-1:i+1],
                y2=1,
                color=color,
                alpha=0.35
            )

ax.set_title("Google Play Store: Total Cumulative Installs Over Time by Category (MoM Growth > 20% Shaded)", fontsize=14, color='white', pad=15)
ax.set_xlabel("Timeline (Year-Month)", fontsize=12, color='white')
ax.set_ylabel("Total Cumulative Installs", fontsize=12, color='white')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=45)
ax.legend(title="Category", loc='upper left', frameon=True, facecolor='#1E1E1E', edgecolor='#333333', labelcolor='white')
ax.grid(True, color='#333333', alpha=0.3)
plt.tight_layout()

# Save figure to Screenshots/Graph1.png
plt.savefig('Screenshots/Graph1.png', dpi=150, facecolor='#121212')
plt.close()